In [0]:
raw_path = "/Volumes/patient_kg_dev/landing/synthea_raw"

for file in dbutils.fs.ls(raw_path):
    print(file.name, file.size)

In [0]:
patients = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .csv(f"{raw_path}/patients.csv")
)

print(patients.count())
patients.printSchema()
display(patients.limit(5))

In [0]:
raw_path = "/Volumes/patient_kg_dev/landing/synthea_raw/patients.csv"

patients_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .csv(raw_path)
)

patients_bronze.printSchema()
print(patients_bronze.count())

In [0]:
from pyspark.sql import functions as F
patients_bronze = patients_bronze.select(
    "*",
    F.col("_metadata.file_name").alias("_source_file"),
    F.col("_metadata.file_path").alias("_source_file_path"),
    F.col("_metadata.file_size").alias("_source_file_size"),
    F.col("_metadata.file_modification_time").alias(
        "_source_file_modified_at"
    ),
)

In [0]:
source_columns = [
    column
    for column in patients_bronze.columns
    if not column.startswith("_")
]
patients_bronze = patients_bronze.withColumn(
    "_row_content_sha256",
    F.sha2(
        F.to_json(
            F.struct(*[F.col(column) for column in source_columns]),
            options={"ignoreNullFields": "false"},
        ),
        256,
    ),
)

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

run_id = str(uuid4())
ingested_at = datetime.now(timezone.utc)
patients_bronze = (
    patients_bronze
    .withColumn("_ingestion_run_id", F.lit(run_id))
    .withColumn("_ingested_at", F.lit(ingested_at))
)

In [0]:
(
    patients_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("patient_kg_dev.bronze.patients")
)

In [0]:
%sql
SELECT COUNT(*)
FROM patient_kg_dev.bronze.patients;

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(Id) AS populated_id_count,
    COUNT(DISTINCT Id) AS distinct_id_count
FROM patient_kg_dev.bronze.patients;

In [0]:
%sql
SELECT DISTINCT PATIENT
FROM patient_kg_dev.bronze.conditions;

SCOPE FOR COUPLE OF PATIENTS ONLY

In [0]:
%sql
SELECT DISTINCT PATIENT
FROM patient_kg_dev.bronze.conditions
WHERE SYSTEM = 'http://snomed.info/sct'
  AND CODE = '414545008';

INGESTION BRONZE LAYER

In [0]:
RAW_VOLUME = "/Volumes/patient_kg_dev/landing/synthea_raw"
BRONZE_SCHEMA = "patient_kg_dev.bronze"

DATASETS = {
    "patients": "patients.csv",
    "encounters": "encounters.csv",
    "conditions": "conditions.csv",
    "medications": "medications.csv",
    "procedures": "procedures.csv",
    "observations": "observations.csv",
    "careplans": "careplans.csv",
}

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F


def ingest_bronze_dataset(
    dataset_name: str,
    source_file: str,
    run_id: str,
):
    source_path = f"{RAW_VOLUME}/{source_file}"
    target_table = f"{BRONZE_SCHEMA}.{dataset_name}"

    source_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "FAILFAST")
        .option("nullValue", "__SOURCE_NULL__")
        .csv(source_path)
    )

    source_columns = source_df.columns

    bronze_df = (
        source_df
        .select(
            "*",
            F.col("_metadata.file_name").alias("_source_file"),
            F.col("_metadata.file_path").alias("_source_file_path"),
            F.col("_metadata.file_size").alias("_source_file_size"),
            F.col("_metadata.file_modification_time").alias(
                "_source_file_modified_at"
            ),
        )
        .withColumn(
            "_row_content_sha256",
            F.sha2(
                F.to_json(
                    F.struct(
                        *[F.col(column) for column in source_columns]
                    ),
                    options={"ignoreNullFields": "false"},
                ),
                256,
            ),
        )
        .withColumn("_ingestion_run_id", F.lit(run_id))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    source_count = source_df.count()

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    target_count = spark.table(target_table).count()

    if source_count != target_count:
        raise RuntimeError(
            f"Count mismatch for {dataset_name}: "
            f"source={source_count}, target={target_count}"
        )

    return {
        "dataset": dataset_name,
        "source_file": source_file,
        "target_table": target_table,
        "source_count": source_count,
        "target_count": target_count,
        "status": "PASS",
    }

In [0]:
run_id = str(uuid4())
results = []

for dataset_name, source_file in DATASETS.items():
    print(f"Ingesting {dataset_name}...")

    result = ingest_bronze_dataset(
        dataset_name=dataset_name,
        source_file=source_file,
        run_id=run_id,
    )

    results.append(result)

print(f"Completed Bronze ingestion run: {run_id}")

In [0]:
results_df = spark.createDataFrame(results)
display(results_df.orderBy("dataset"))

**Metadata Validation**

In [0]:
SHOW TABLES IN patient_kg_dev.bronze;

In [0]:
%sql
DESCRIBE DETAIL patient_kg_dev.graph_ready.patient_nodes;

In [0]:
display(
    spark.sql("""
        DESCRIBE DETAIL patient_kg_dev.graph_ready.patient_nodes
    """).select(
        "format",
        "location",
        "numFiles",
        "sizeInBytes"
    )
)

In [0]:
graph_tables = [
    "cohort_index",
    "patient_nodes",
    "encounter_nodes",
    "condition_nodes",
    "medication_nodes",
    "procedure_nodes",
    "node_index",
    "relationships"
]

for table_name in graph_tables:
    detail = (
        spark.sql(
            f"DESCRIBE DETAIL patient_kg_dev.graph_ready.{table_name}"
        )
        .select("location")
        .first()
    )

    print(table_name, "->", detail["location"])

In [0]:
CATALOG = "patient_kg_dev"
GRAPH = f"{CATALOG}.graph_ready"

graph_tables = [
    "cohort_index",
    "patient_nodes",
    "encounter_nodes",
    "condition_nodes",
    "medication_nodes",
    "procedure_nodes",
    "node_index",
    "relationships"
]

for table_name in graph_tables:
    detail = (
        spark.sql(
            f"DESCRIBE DETAIL {GRAPH}.{table_name}"
        )
        .select(
            "name",
            "format",
            "location"
        )
        .first()
    )
    print(
        table_name,
        "| format =", detail["format"],
        "| location =", repr(detail["location"])
    )

In [0]:
display(dbutils.secrets.listScopes())

In [0]:
display(
    dbutils.secrets.list("healthcare-graphrag")
)